# Selective-Layer LoRA: gradient-selected adapters, measured honestly
Qwen2.5-0.5B-Instruct · MultiAlpaca (ar/ru/th) · PEFT LoRA · free Colab T4

**Research question.** Can gradient-based layer selection cut peak GPU memory and trainable parameters while staying close to all-layer LoRA in quality? We **measure** it; nothing here assumes the answer.

**Run the cells top to bottom.** Each cell says what it does, what output to expect, and what to do if it fails. All heavy code lives in the repo (`sllora/`, `scripts/`); these cells drive it and show the results.

| Phase | Cells | GPU time |
|---|---|---|
| 1 Environment | 1-4 | ~3 min |
| 2 Data | 5-6 | CPU |
| 3 Gradient scoring | 7 | ~3 min |
| 4 Selective LoRA check | 8 | ~1 min |
| 5 Smoke test + memory profiling | 9-10 | ~6 min |
| 6 Experiments A-E | 11 | ~70 min (split over 2 accounts with `--only`) |
| 7-8 Evaluation + Pareto | 12-13 | CPU |

## Phase 1: environment
**Cell 1** checks Python, PyTorch, CUDA, the GPU, GPU memory and system RAM.
*Expect:* `Tesla T4`, about 15 GB GPU memory, about 12 GB RAM. *If `cuda False`:* Runtime → Change runtime type → T4 GPU.

In [ ]:
import sys, torch, psutil, platform
print("python", sys.version.split()[0], "| torch", torch.__version__, "| cuda", torch.version.cuda, "| cuda available", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print("GPU", p.name, f"| {p.total_memory/1024**3:.1f} GB")
print(f"system RAM {psutil.virtual_memory().total/1024**3:.1f} GB total, {psutil.virtual_memory().available/1024**3:.1f} GB free")

**Cell 2** mounts the shared Drive and gets the code from GitHub, then installs only what Colab lacks. Torch is not reinstalled.
*Expect:* `ALL SMOKE TESTS PASSED` and `ALL SPEC FEATURE TESTS PASSED`. *If the clone fails:* check that the `GH_TOKEN` secret exists and has Notebook access turned on.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os
t = userdata.get('GH_TOKEN')
if not os.path.exists('/content/repo'):
    !git clone -q https://{t}@github.com/SumanGiri110/new-llm.git /content/repo
%cd /content/repo
!git pull -q
!pip uninstall -y -q torchao   # Colab's old torchao breaks recent peft; we don't use it
!pip install -q -r requirements.txt
!python tests/smoke_test.py 2>&1 | tail -1
!python tests/test_spec_features.py 2>&1 | tail -1
!mkdir -p results/runs && cp -n /content/drive/MyDrive/hashcode_sllora/results/*.json results/runs/ 2>/dev/null; echo synced

**Cell 3** loads the model, prints the layer count and the real module names, and measures GPU memory before and after loading.
*Expect:* 24 layers, with `q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj` inside each layer, and about 0.95 GB for fp16 weights. The target names in Phase 4 come from this printout and are not assumed.

In [ ]:
from sllora.model import load_model_and_tokenizer, n_layers_of
from sllora.utils import free_cuda
free_cuda(); before = torch.cuda.memory_allocated()/1024**3
model, tok = load_model_and_tokenizer("Qwen/Qwen2.5-0.5B-Instruct")
after = torch.cuda.memory_allocated()/1024**3
print("layers:", n_layers_of(model))
print("modules in layer 0:", sorted({n.split('.')[-1] for n, m in model.model.layers[0].named_modules() if n}))
print(f"GPU memory: {before:.2f} GB before load -> {after:.2f} GB after load")
del model; free_cuda()

**Cell 4** saves the package versions and the experiment configuration so the run can be reproduced.

In [ ]:
from sllora.train import library_versions, RunConfig
from dataclasses import asdict
import json
env = {"versions": library_versions(), "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
       "default_config": asdict(RunConfig(run_id="template"))}
os.makedirs("results", exist_ok=True); json.dump(env, open("results/environment.json", "w"), indent=2)
print(json.dumps(env["versions"], indent=1))

## Phase 2: MultiAlpaca
**Dataset:** `DAMO-NLP-MT/multialpaca` (Apache-2.0). It has one JSONL file per language with fields `instruction`, `input` (often null) and `output`. We verify this below rather than assuming it.

**Cell 5** inspects the schema and a few examples.

In [ ]:
from sllora import data as D
rows = D.load_multialpaca_lang("ar")
print(len(rows), "Arabic rows | fields:", list(rows[0].keys()))
for r in rows[:2]: print(json.dumps(r, ensure_ascii=False)[:300])
for lang in ["ru", "th"]: print(lang, len(D.load_multialpaca_lang(lang)), "rows")

**Cell 6** builds the splits and checks for leakage. Training uses 1,000 per language for the experiment A-E suite. Validation uses the same 200 per language as the main sweep. Scoring uses 43 per language, **disjoint from both**. The cell then shows how the chat template masks the prompt tokens.
*Expect:* `overlap 0` three times, and a masked-label count greater than 0.

In [ ]:
sp = D.build_splits(n_train=1000, n_eval=200, cache_path="data/splits_spec.json")
sc = D.build_scoring_set(n_per_lang=43)
key = lambda e: (e["lang"], e["instruction"], e["output"][:80])
tr = {key(e) for e in sp["train"]}; ev = {key(e) for l in sp["eval"].values() for e in l}; scs = {key(e) for e in sc}
print("train", len(tr), "| eval", len(ev), "| scoring", len(scs))
print("overlap train/eval", len(tr & ev), "| train/scoring", len(tr & scs), "| eval/scoring", len(ev & scs))
from sllora.model import load_model_and_tokenizer
_, tok = load_model_and_tokenizer("Qwen/Qwen2.5-0.5B-Instruct", device="cpu")
x = D.tokenize_example(tok, sp["train"][0], max_len=256)
print("tokens", len(x["input_ids"]), "| prompt tokens masked (-100):", sum(1 for l in x["labels"] if l == -100))
items = D.tokenize_all(tok, sp["train"], 256)
print("kept", len(items), "of", len(sp["train"]), "after truncation/empty-answer filtering")

## Phase 3: gradient-based layer scoring
**Protocol** (`sllora/scoring.py: layer_importance_protocol`):
1. Load a fresh model and attach **temporary** rank-4 LoRA to `q_proj` and `v_proj` in all 24 layers, with dropout 0 and a fixed seed.
2. For each of about 32 batches of 4 scoring examples: compute the response-token loss, run backward, and record the per-layer score `s_l = ||grad B_l||_2` over that layer's LoRA-B matrices.
3. Average over batches: `S_l = mean_b s_l(b)`. Also report the std, and a **split-half Spearman** correlation (do two halves of the batches rank layers the same way?).
4. Remove the adapters with `unload()`. No temporary weights or optimizer state reach training, because the training runs load a fresh model.

**Why B and not A:** LoRA initialises B = 0, so the gradient of A is exactly 0 at the start. The gradient of B is `(dL/dh)(Ax)^T`, a random projection of the full weight gradient. **Limitations:** it depends on the random A, sees only q/v, and is measured at initialisation. It's a heuristic for where learning signal is, not proof that those layers will validate better.

**Cell 7** runs it. *Expect:* 24 scores, the top-6 layers, and a split-half Spearman correlation, where values near 1 mean a stable ranking.

In [ ]:
!python scripts/score_layers.py --protocol
s = json.load(open("results/scores/layer_scores.json"))["importance_qv_r4"]
import matplotlib.pyplot as plt, numpy as np
m, sd = np.array(s["raw_mean"]), np.array(s["raw_std"])
plt.figure(figsize=(10, 3)); plt.bar(range(24), m / m.max(), yerr=sd / m.max(), color="#e87ba4", capsize=2)
plt.xlabel("layer"); plt.ylabel("normalised score"); plt.title(f"rank-4 q/v gradient score, {s['n_batches']} batches (error bar = std)", loc="left"); plt.show()
print("top-6:", sorted(s["ranking"][:6]), "| split-half Spearman:", round(s["split_half_spearman"], 3))
!git add -f results/scores && git commit -qm "scores: rank-4 q/v gradient protocol" ; git push -q

## Phase 4: selective LoRA construction (verification only)
**Cell 8** builds the top-6 model the way training does: a fresh base model, new rank-16 LoRA on `q_proj` and `v_proj` of the 6 selected layers, and everything else frozen. It then checks which parameters are trainable.
*Expect:* only `lora_A`/`lora_B` parameters, about 0.03% trainable, and the optimizer holding exactly those parameters.

In [ ]:
from sllora.model import apply_selective_lora, adapted_layers, count_params
from sllora.selection import select_layers, layer_strip
top6 = select_layers("importance", 24, k=6, scores={"importance": s["scores"]})
m0, tok = load_model_and_tokenizer("Qwen/Qwen2.5-0.5B-Instruct")
pm = apply_selective_lora(m0, top6, r=16, alpha=32, dropout=0.05, targets=["q_proj", "v_proj"])
tr, tot = count_params(pm)
print("layers", layer_strip(top6, 24), "| adapted:", adapted_layers(pm))
print(f"trainable {tr:,} / {tot:,} = {100*tr/tot:.4f}%")
print("non-LoRA trainable:", [n for n, p in pm.named_parameters() if p.requires_grad and "lora_" not in n] or "none")
opt = torch.optim.AdamW([p for p in pm.parameters() if p.requires_grad]); print("optimizer params:", sum(p.numel() for g in opt.param_groups for p in g["params"]))
del pm, m0, opt; free_cuda()

## Phase 5: smoke test, then memory profiling
**Cell 9** runs a 2-step training smoke test with the experiment B config: 4-bit, q/v, top-6, sequence length 256, micro-batch 1 × 8 accumulation. The run itself checks for a NaN loss, **missing gradients on any LoRA parameter**, and invalid layers, and records any out-of-memory error. It then deletes its outputs.
*Expect:* `step 2/2`, then `done`.

In [ ]:
!python -m sllora.train --run_id smoke_spec --rule importance --k 6 --score_key importance_qv_r4 --targets q_proj,v_proj \
   --quant4 --chunked --max_len 256 --bs 1 --accum 8 --n_train 50 --n_eval 10 --max_steps 2 --eval_suite none \
   --probe_bs 1 --probe_len 256 --split_cache data/splits_smoke.json --out_dir /content/smoke --results_dir /content/smoke_res
!rm -rf /content/smoke /content/smoke_res data/splits_smoke.json

**Cell 10: memory profile.** Memory-only runs for A, B, C and D. Each measures peak allocated and reserved memory plus the bytes autograd keeps for backward, and each runs in its **own process**. This tells us whether activations still matter after checkpointing, which is the test for whether offloading is worth it.
*Read it like this:* if C's `saved_act` is already tiny, offloading (D) can only save a little. Offloading also costs time and CPU RAM, and is measured in Phase 6.

In [ ]:
import subprocess, glob
common = "--targets q_proj,v_proj --quant4 --chunked --max_len 256 --probe_bs 1 --probe_len 256 --train_only_probe --score_key importance_qv_r4 --results_dir /content/probe_res --out_dir /content/probe_out"
cfgs = {"A_all": "--rule all", "B_grad6": "--rule importance --k 6", "C_grad6_gc": "--rule importance --k 6 --gc",
        "D_grad6_gc_offload": "--rule importance --k 6 --gc --offload_acts"}
for name, extra in cfgs.items():
    subprocess.run(f"SLLORA_ISOLATED=1 python -m sllora.train --run_id prof_{name} {extra} {common}", shell=True, capture_output=True)
for f in sorted(glob.glob("/content/probe_res/*.json")):
    r = json.load(open(f)); m = r["memory"]
    print(f"{r['run_id']:24s} peak {m['probe_peak_gb']:.3f} GB  reserved {m['probe_reserved_gb']:.3f} GB  saved_act/256tok {m['saved_act_gb_per_256tok']:.4f} GB  trainable {r['trainable_pct']}%")

## Phase 6: experiments A-E (same model, data, sequence length and evaluation)
| Exp | run_id | What |
|---|---|---|
| ref | `spec_ref` | untouched model (the before) |
| A | `spec_A_all` | all-layer LoRA, q/v, r16 |
| B | `spec_B_grad6` | gradient-selected top-6 |
| C | `spec_C_grad6_gc` | B + gradient checkpointing |
| D | `spec_D_grad6_gc_offload` | C + CPU activation offloading |
| E | `spec_E_grad3`, `spec_E_grad12` | layer-count ablation (24 = A) |

All runs use a 4-bit base, chunked loss (identical loss), lr 2e-4, 1 epoch, clipping 1.0 and seed 42. Every run is a fresh process from the same pretrained checkpoint, saves a resumable checkpoint every 100 steps, and **records out-of-memory errors as `status: oom`** instead of dropping them.

**Cell 11.** It's resumable: if Colab disconnects, re-run it. To split over two accounts, use `--only spec_ref,spec_A_all,spec_B_grad6` on one and `--only spec_C_grad6_gc,spec_D_grad6_gc_offload,spec_E_grad3,spec_E_grad12` on the other.

In [ ]:
!python scripts/run_sweep.py --queue SPEC --out /content/drive/MyDrive/hashcode_sllora/outputs --drive_results /content/drive/MyDrive/hashcode_sllora/results --git_push

## Phases 7-8: evaluation table and Pareto frontier
**Cell 12** builds the table and frontier for the A-E suite. Memory reduction = (baseline − experiment) / baseline × 100, against `spec_A_all`. The Pareto frontier uses **measured** memory only (minimise) against held-out loss (minimise). Δtask is measured against `spec_ref`. Outputs: `results/pareto_spec.csv`, `results/pareto_spec.md` and `results/plots/pareto_spec.png`.

In [ ]:
!git pull -q; cp -n /content/drive/MyDrive/hashcode_sllora/results/*.json results/runs/ 2>/dev/null
!python scripts/pareto.py --suite spec
from IPython.display import Image, display; display(Image("results/plots/pareto_spec.png"))

**Cell 13** does the same for the main sweep (7 selection rules, memory stacks), then commits everything.

In [ ]:
!python scripts/pareto.py --suite main
display(Image("results/plots/pareto_main.png"))
!git add -f results && git commit -qm "results: pareto analysis" ; git push -q

## Troubleshooting
| Symptom | Fix |
|---|---|
| `Found an incompatible version of torchao` | `pip uninstall -y torchao` (cell 2 does this) |
| `Repository not found` on clone | `GH_TOKEN` secret missing, Notebook access off, or not a collaborator |
| `CUDA out of memory` | It's recorded in the run's JSON as `status: oom`. Lower `--bs` / `--max_len`, or add `--gc` |
| Loss NaN 5× → run stops | `--lr 1e-4`; fp16 overflow is rare at 0.5B |
| `score 'importance_qv_r4' not in ...` | Run cell 7 (`score_layers.py --protocol`) and `git pull` on other accounts |
| Memory row says `inflated` | `python scripts/reprobe.py --auto` (re-measures in a fresh process) |
| Colab disconnected mid-run | Re-run the same cell. Training resumes from `ckpt.pt`, and finished runs are skipped |
| GPU allowance used up | Move accounts (share the Drive folder, add the shortcut, set the `GH_TOKEN` secret), then use `--only` for the remaining runs |

**Honesty rules:** report only measured values; mark estimates as estimates; compare runs only when their settings match; call differences smaller than the seed-to-seed gap noise.